In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import os

/opt/anaconda3/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
file = 'final_prompts.csv'

In [4]:
# Defining reading method for different encodings and separators
def read_csv(filepath):
     if os.path.splitext(filepath)[1] != '.csv':
          return  
     seps = [',', ';', '\t']                   
     encodings = [None, 'utf-8', 'ISO-8859-1', 'utf-16','ascii']  
     for sep in seps:
         for encoding in encodings:
              try:
                  return pd.read_csv(filepath, encoding=encoding, sep=sep)
              except Exception:  
                  pass
     raise ValueError("{!r} is has no encoding in {} or seperator in {}"
                      .format(filepath, encodings, seps))

In [5]:
data_df = read_csv(file)

In [6]:
#Dropping lines with null values and changing indeces to be consistent
data_df_cleaned = data_df.dropna(how='all')
data_df_cleaned['Model'] = data_df_cleaned['Model'].replace('Mistral-7B-v0.1', 'mistral-7b')


/var/folders/mj/_qhvxt117n1fnjzlrt6kt408k5_y3g/T/ipykernel_80113/2571352431.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_df_cleaned['Model'] = data_df_cleaned['Model'].replace('Mistral-7B-v0.1', 'mistral-7b')


In [7]:
#Loading transformer model
model = SentenceTransformer('bert-base-nli-mean-tokens')
#Intialising part of reference text as answer aka original
original_text = 'Listening tests are specifically designed to evaluate audio quality e.g. for new audio technologies or an updated version'


/opt/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
data_df_cleaned.columns = ['index', 'answer']

In [9]:
#Encoding original text as vector
org_vector = model.encode(original_text, convert_to_tensor= True)

In [10]:
#Defining method to calculate cosine similarity
def calculate_similarity(answer):
    answer_vector = model.encode(answer, convert_to_tensor=True)
    similarity_score = util.pytorch_cos_sim(org_vector, answer_vector)
    return similarity_score.item()

In [11]:
#Calculating cosine scores and adding them to new column
data_df_cleaned['cos_score'] = data_df_cleaned['answer'].apply(calculate_similarity)

/var/folders/mj/_qhvxt117n1fnjzlrt6kt408k5_y3g/T/ipykernel_80113/505916793.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_df_cleaned['cos_score'] = data_df_cleaned['answer'].apply(calculate_similarity)


In [12]:
output_file_path = "output.tsv"
data_df_cleaned.to_csv(output_file_path, sep='\t', index=False)

In [13]:
min_scores = data_df_cleaned.loc[data_df_cleaned.groupby('index')['cos_score'].idxmin()]
print(min_scores)

             index                                             answer  \
72  ChatGPT-4-0613  Listening tests are conducted to evaluate and ...   
6      llama-2-70b  2 points: * Useful: Entertaining: 3 points: 1 ...   
18   luminous-base                              :\nA.\nB.\nC.\nD.\nE.   
11      mistral-7b  1. to determine whether a certain attribute, e...   

    cos_score  
72   0.840342  
6    0.684872  
18   0.242462  
11   0.527342  


In [14]:
min_scores.to_csv('min_scores.tsv', sep='\t', index=False)

In [15]:
max_scores = data_df_cleaned.loc[data_df_cleaned.groupby('index')['cos_score'].idxmax()]
print(max_scores)

             index                                             answer  \
28  ChatGPT-4-0613  Listening tests are conducted to evaluate audi...   
38     llama-2-70b  "The purpose of listening tests is to assess t...   
26   luminous-base  :\n.2.3.1  Listening Tests for New Audio Techn...   
49      mistral-7b  "Listening tests are conducted to evaluate aud...   

    cos_score  
28   0.941727  
38   0.951986  
26   0.939693  
49   0.938632  
